In [61]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

 # 1. Data Loading

In [62]:
# Load data
df = pd.read_csv('./diabetes.csv')

# Display basic information with clear separators
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)

print(f"\nDataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")

print("\n" + "-" * 50)
print("DESCRIPTIVE STATISTICS")
print("-" * 50)
print(df.describe().T.round(2))  # Rounded for readability

print("\n" + "-" * 50)
print("FIRST 5 ROWS")
print("-" * 50)
print(df.head())

print("\n" + "-" * 50)
print("DATA INFO")
print("-" * 50)
print(df.info())

DATASET OVERVIEW

Dataset Shape: 768 rows × 9 columns

--------------------------------------------------
DESCRIPTIVE STATISTICS
--------------------------------------------------
                          count    mean     std    min    25%     50%     75%  \
Pregnancies               768.0    3.85    3.37   0.00   1.00    3.00    6.00   
Glucose                   768.0  120.89   31.97   0.00  99.00  117.00  140.25   
BloodPressure             768.0   69.11   19.36   0.00  62.00   72.00   80.00   
SkinThickness             768.0   20.54   15.95   0.00   0.00   23.00   32.00   
Insulin                   768.0   79.80  115.24   0.00   0.00   30.50  127.25   
BMI                       768.0   31.99    7.88   0.00  27.30   32.00   36.60   
DiabetesPedigreeFunction  768.0    0.47    0.33   0.08   0.24    0.37    0.63   
Age                       768.0   33.24   11.76  21.00  24.00   29.00   41.00   
Outcome                   768.0    0.35    0.48   0.00   0.00    0.00    1.00   

         

# 2. Data Preprocessing

## Step 1: Handle missing/zero values (medically impossible zeros)

In [63]:
print(f"Missing values: {df.isnull().sum().sum()}")

# Check zeros in medical features
medical_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in medical_features:
    zero_count = (df[col] == 0).sum()
    print(f"  {col}: {zero_count} zeros")

# Replace zeros with NaN for imputation
df_processed = df.copy()
for col in medical_features:
    df_processed.loc[df_processed[col] == 0, col] = np.nan
print("Zeros converted to NaN for proper imputation")


Missing values: 0
  Glucose: 5 zeros
  BloodPressure: 35 zeros
  SkinThickness: 227 zeros
  Insulin: 374 zeros
  BMI: 11 zeros
Zeros converted to NaN for proper imputation


## Step 2: Handle Outliers

In [64]:
# handle outliers function
print("\nHandling outliers in medical features")
def handle_outliers(data, column):
    non_nan_data = data[column].dropna()

    if len(non_nan_data) == 0:
        return

    Q1 = non_nan_data.quantile(0.25)
    Q3 = non_nan_data.quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    mask = ~data[column].isna()
    data.loc[mask, column] = data.loc[mask, column].clip(lower_bound, upper_bound) .round().astype(int)

# Handle outliers in key features (especially Insulin which has many zeros)
features_for_outliers = ['Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Pregnancies']
for feature in features_for_outliers:
    handle_outliers(df_processed, feature)


Handling outliers in medical features


## Step 3: Feature Engineering - Create BMI and Glucose categories

In [65]:
def create_bmi_category(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif 18.5 <= bmi < 25:
        return 'Normal'
    elif 25 <= bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

def create_glucose_category(glucose):
    if glucose < 100:
        return 'Normal'
    elif 100 <= glucose < 126:
        return 'Prediabetic'
    else:
        return 'Diabetic'

# Apply categorization
df_processed['BMI_Category'] = df_processed['BMI'].apply(create_bmi_category)
df_processed['Glucose_Category'] = df_processed['Glucose'].apply(create_glucose_category)


print("BMI amd Glucose categories created successfully")
df_processed.head()


BMI amd Glucose categories created successfully


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,BMI_Category,Glucose_Category
0,6,148.0,72.0,35.0,NaN,34.0,1.0,50,1,Obese,Diabetic
1,1,85.0,66.0,29.0,NaN,27.0,0.0,31,0,Overweight,Normal
2,8,183.0,64.0,NaN,NaN,23.0,1.0,32,1,Normal,Diabetic
3,1,89.0,66.0,23.0,94.0,28.0,0.0,21,0,Overweight,Normal
4,0,137.0,40.0,35.0,168.0,43.0,1.0,33,1,Obese,Diabetic


## Step 4: Define Feature Groups (Before pipeline creation)

In [66]:
numeric_features = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
                    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

categorical_features = ['BMI_Category', 'Glucose_Category']

## Step 5: Split Data

In [67]:
X = df_processed.drop('Outcome', axis=1)
y = df_processed['Outcome']

# Split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Step 6: Create Preprocessing Pipeline

In [68]:
# Numerical pipeline: Impute → Scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: Impute → Encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

# Combined pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

## Step 7: Fit and Transform Data

In [69]:
print("Fitting pipeline on training data...")
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data using fitted pipeline
print("Transforming test data...")
X_test_processed = preprocessor.transform(X_test)



# Get feature names
feature_names = numeric_features.copy()
for i, col in enumerate(categorical_features):
    categories = preprocessor.named_transformers_['cat'].named_steps['encoder'].categories_[i]
    for cat in categories[1:]:  # Skip first (drop='first')
        feature_names.append(f"{col}_{cat}")

print("feature names:", feature_names)



Fitting pipeline on training data...
Transforming test data...
feature names: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'BMI_Category_Obese', 'BMI_Category_Overweight', 'BMI_Category_Underweight', 'Glucose_Category_Normal', 'Glucose_Category_Prediabetic']
